<!-- GENERATED by scripts/build_notebooks.py from _variables.yml. Do not edit this cell. -->
# 9 · Debugging and evaluating RL

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/practical-rl/blob/main/notebooks/09-debugging-evaluation.ipynb)

**Practical Reinforcement Learning** · Day 3 · Module 9 · briefing 14:45, lab 15:25 · core 52 min · designed for Google Colab, CPU runtime

Diagnose broken agents from their logs, confirm with probe environments, and report results honestly.

**By the end of this notebook you can:**

- Diagnose four DQN failures from their logged metrics by writing the signal that exposes each one.
- Confirm a diagnosis with probe environments and a debugging checklist.
- Report a comparison with IQM, bootstrap confidence intervals and a budget sentence, and call ties.
- Interpret a hyperparameter sensitivity sweep.

**How the lab works.** Each exercise: *Predict* (write your guess first) → a `# TODO` cell you complete → *Run* → *Explain* → a checkpoint that passes or fails. A folded Hint and Solution sit under each TODO. The solution never replaces your code; if you are stuck, run `lab.use_reference(N)` to continue with it.

Briefing: [Module 9 · In the lab](https://project-delphi.github.io/practical-rl/modules/09-debugging-evaluation.html#in-the-lab)

**Next:** [Module 10 · Policy gradients](https://project-delphi.github.io/practical-rl/modules/10-policy-gradients.html)

In [ ]:
#@title Setup: run this cell first { display-mode: "form" }
# GENERATED by scripts/build_notebooks.py. On Colab it installs exactly what this lab adds
# (keeping every preinstalled package); elsewhere it checks versions and warns.
import importlib.metadata as _md
import os as _os
import subprocess as _sp
import sys as _sys

PRL_REF = "main"  #@param {type:"string"}
WORKED = False  #@param {type:"boolean"}
SEED = 0  #@param {type:"integer"}

_PINS = {"gymnasium": "1.4.0"}
_ALLOW = ["gymnasium"]
_CORE = ["torch", "numpy", "scipy", "pandas", "matplotlib", "transformers", "accelerate", "datasets"]
_PRL_SPEC = _os.environ.get("PRL_SPEC") or "git+https://github.com/project-delphi/practical-rl@" + PRL_REF + "#subdirectory=prl"
_on_colab = ("google.colab" in _sys.modules or bool(_os.environ.get("COLAB_RELEASE_TAG"))
             or _os.environ.get("PRL_PLATFORM") == "colab-sim")


def _norm(name):
    return name.lower().replace("_", "-").replace(".", "-")


def _installed_pins(names=None):
    """One name==version per package: the copy Python imports (first on sys.path).

    Colab's sys.path also holds older system copies of some packages; pinning
    every copy would ask pip for two versions of the same package.
    """
    pins = {}
    for d in _md.distributions():
        name = d.metadata["Name"]
        key = _norm(name) if name else None
        if not key or key in pins or key in _keep or (names is not None and key not in names):
            continue
        try:
            pins[key] = f"{name}=={_md.version(name)}"
        except _md.PackageNotFoundError:
            continue
    return sorted(pins.values())


def _pip_install(constraints):
    with open("/tmp/prl-constraints.txt", "w") as fh:
        fh.write("\n".join(constraints) + "\n")
    cmd = [_sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/prl-constraints.txt",
           *[f"{k}=={val}" for k, val in _PINS.items()], _PRL_SPEC]
    return _sp.run(cmd, capture_output=True, text=True)


if _on_colab:
    _keep = set(_ALLOW) | {"prl"}
    _before = {k: _md.version(k) for k in ("numpy", "torch", "scipy") if k in _sys.modules}
    _r = _pip_install(_installed_pins())
    if _r.returncode != 0:
        # Fall back to protecting only the heavy preinstalled packages.
        print("Keeping every preinstalled package was not possible; keeping the core ones "
              f"({', '.join(_CORE)}) and retrying.")
        print(_r.stderr.strip().splitlines()[-1] if _r.stderr.strip() else "")
        _r = _pip_install(_installed_pins(set(_CORE)))
    if _r.returncode != 0:
        print(_r.stdout[-3000:], _r.stderr[-3000:])
        raise SystemExit("Setup failed: the runtime's packages changed. Tell the instructor.")
    if any(_md.version(k) != val for k, val in _before.items()):
        print("A package that was already imported changed version: Runtime → Restart session, "
              "then run this cell again.")
else:
    for _name, _want in _PINS.items():
        try:
            _have = _md.version(_name)
        except _md.PackageNotFoundError:
            _have = None
        if _have != _want:
            print(f"Warning: this lab expects {_name} {_want}, found {_have}. "
                  "Run `uv sync --group notebooks` in the repository.")

import prl
import prl.plot
import prl.runtime

prl.runtime.print_versions(extra=list(_PINS))
prl.plot.style()
print(f"SEED = {SEED}")

In [ ]:
# GENERATED by scripts/build_notebooks.py. Starts the lab harness; rerunning it is safe.
import prl.lab

lab = prl.lab.init("09-debugging-evaluation", "cf6052886e6fd869", designed="colab-cpu", api=1, ns=globals(),
                   checkpoints=["1.1", "1.2", "2.1", "2.2", "3.1", "3.2", "3.3", "3.4", "4.1", "4.2", "5.1", "5.2", "6.1", "6.2"], packages=["gymnasium"], seat=SEED, badge_ref=PRL_REF,
                   worked=WORKED or None)

A teammate's DQN on CartPole has stopped learning. In this lab you read the recorded logs of
five runs of Module 8's reference DQN at the same budget: one healthy run and four broken
ones, A to D, each recorded on 3 seeds. You write four signals and try them on the healthy
run, then diagnose the four broken runs from all four signals at once. You write two probe
environments and run every agent on them, to test the diagnoses. Last, you report a recorded
learning-rate sweep with the interquartile mean (IQM) and bootstrap intervals, and place one
live seed of your own against its band.

The briefing's equations are named in brackets, for example (TD error) or (IQM). Its table
"Equations and where you implement them" gives each name's equation number.

In [ ]:
import dataclasses
import textwrap
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch

import prl.data
from prl.checks import m09  # the checkpoints of this lab
from prl.evaluate import Budget, compare

# Budgets (live, then QUICK). Lab code reads them from here and never branches on QUICK.
# `steps`, `log_every`, `eval_every` and `eval_episodes` are Exercise 6's live run and pick its
# recorded band; the recorded runs of Exercises 1-3 and 5 used the live values. `probe_steps`
# is each probe run of Exercise 4.
budget = lab.use_settings(
    prl.runtime.settings(
        dict(steps=50_000, log_every=500, eval_every=5_000, eval_episodes=10, probe_steps=2_000),
        dict(steps=10_000, log_every=500, eval_every=1_000, eval_episodes=10, probe_steps=2_000),
        designed="colab-cpu",
    )
)
# Your seat's seed, clear of the recorded seeds, the figures' and the checkpoints' (0-99).
RUN_SEED = (SEED + 1) * 1000
_rt = prl.runtime.detect()
HARDWARE = f"{_rt.platform}, {_rt.cpu}, {_rt.n_cpu} CPU"
torch.set_num_threads(min(2, _rt.n_cpu))
print(f"Budget: {budget.steps:,} env steps for your live run; {budget.probe_steps:,} per probe.")

### The reference version of your Lab 8 code

The two cells below are Module 8's DQN: its configuration `CONFIG`, the masked target
`td_target`, `gather_q`, which reads Q(s, a; w) for the action taken, and the training loop
`train_dqn`, which returns the network and a log. Every run in this lab, recorded or live,
uses this code and this configuration.

In [ ]:
# Reference code included from labs/shared/dqn_setup.py.
# origin: labs/src/08-dqn.py
# Provided code for the workshop's one DQN (Lab 8): the configuration, the network,
# epsilon-greedy acting, the gather, the Huber loss, the DQN target, the logger, train_dqn's
# setup (dqn_parts) and greedy evaluation. Lab 8 includes this file, and so do Modules 9, 14
# and 15b, so every DQN in the workshop runs the same configuration. Two names are read from
# the notebook each time they run: dqn_target calls td_target (Lab 8, Exercise 2) and
# dqn_parts builds a ReplayBuffer (Lab 8, Exercise 1).
import dataclasses
import time
import types
from typing import NamedTuple

import numpy as np
import torch
from torch import nn

PROBE_SEED = 1_000_000  # probe states: reset(seed=PROBE_SEED + i)
EVAL_SEED = 1_100_000  # evaluation episodes: reset(seed=EVAL_SEED + i)


@dataclasses.dataclass(frozen=True)
class DQNConfig:
    """The DQN configuration every lab uses. Change a copy with dataclasses.replace."""

    hidden: tuple = (64, 64)  # two hidden layers, ReLU
    lr: float = 1e-3  # Adam's step size
    buffer_size: int = 10_000  # replay capacity, in transitions
    batch_size: int = 64
    gamma: float = 0.99
    train_freq: int = 4  # environment steps per gradient step
    target_sync: int | None = 500  # steps between copies w -> w-; None: never
    learning_starts: int = 1_000  # gradient steps start after this step
    eps_start: float = 1.0
    eps_end: float = 0.05
    eps_fraction: float = 0.2  # epsilon falls linearly over this share of the run
    max_grad_norm: float = 10.0  # clip the gradient's total norm to this
    huber_threshold: float = 1.0  # psi in the Huber loss


CONFIG = DQNConfig()


class Batch(NamedTuple):
    """A minibatch of B transitions as tensors."""

    obs: torch.Tensor  # (B, obs_dim) float32
    act: torch.Tensor  # (B,) int64
    rew: torch.Tensor  # (B,) float32
    next_obs: torch.Tensor  # (B, obs_dim) float32
    terminated: torch.Tensor  # (B,) float32: 1.0 if s' is a terminal state
    truncated: torch.Tensor  # (B,) float32: 1.0 if the episode was cut at s'


def to_batch(buffer, idx):
    """The transitions in slots idx of a replay buffer, as a Batch of tensors."""
    cols = (buffer.obs, buffer.act, buffer.rew, buffer.next_obs, buffer.terminated)
    return Batch(*(torch.as_tensor(c[idx]) for c in (*cols, buffer.truncated)))


def make_q_net(obs_dim, n_actions, config):
    """Q(s, .; w): obs_dim inputs, config.hidden ReLU layers, one output per action."""
    layers, width = [], obs_dim
    for h in config.hidden:
        layers += [nn.Linear(width, h), nn.ReLU()]
        width = h
    layers.append(nn.Linear(width, n_actions))
    return nn.Sequential(*layers)


def epsilon_at(t, steps, config):
    """Epsilon at step t of a run of `steps`: linear from eps_start to eps_end, then flat."""
    frac = min(1.0, t / max(1.0, config.eps_fraction * steps))
    return config.eps_start + frac * (config.eps_end - config.eps_start)


def act(q_net, obs, epsilon, n_actions, rng):
    """Epsilon-greedy: a uniform action with probability epsilon, else argmax_a Q(s, a; w)."""
    if rng.random() < epsilon:
        return int(rng.integers(n_actions))
    with torch.no_grad():
        q = q_net(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0))
    return int(q.argmax(dim=1)[0])


def gather_q(q_values, actions):
    """Q(s_i, a_i; w) for each row i: q_values (B, |A|) and actions (B,) give (B,)."""
    return q_values.gather(1, actions.long().unsqueeze(1)).squeeze(1)


def huber(x, threshold=1.0):
    """The Huber loss of every entry of x: x^2/2 where |x| <= threshold, and
    threshold * (|x| - threshold/2) beyond. Its derivative is x clipped to [-threshold, threshold]."""
    a = x.abs()
    quadratic = 0.5 * x**2
    linear = threshold * (a - 0.5 * threshold)
    return torch.where(a <= threshold, quadratic, linear)


def dqn_target(batch, q_net, target_net, gamma):
    """DQN's target: td_target with the target network's max_a' Q(s', a'; w-)."""
    with torch.no_grad():
        q_next = target_net(batch.next_obs).max(dim=1).values
        return td_target(batch, q_next, gamma)  # (masked target)


def probe_states(make_env, n=32):
    """n fixed start states, the same for every run: reset(seed=PROBE_SEED + i)."""
    env = make_env()
    obs = [env.reset(seed=PROBE_SEED + i)[0] for i in range(n)]
    env.close()
    return np.asarray(obs, dtype=np.float32)


def evaluate_greedy(q_net, make_env, episodes=10, gamma=0.99):
    """Greedy episodes (epsilon = 0) from fixed start states, reset(seed=EVAL_SEED + i).

    Returns a dict: "returns" (undiscounted), "disc_returns" (sum of gamma^t r_t from the
    start state), "q0" (max_a Q(s0, a; w)), "terminated" (bool), each of shape (episodes,),
    and "rewards", a list with one float array per episode.
    """
    env = make_env()
    out = {k: [] for k in ("returns", "disc_returns", "q0", "terminated", "rewards")}
    for i in range(episodes):
        obs, _ = env.reset(seed=EVAL_SEED + i)
        rewards, done, first = [], False, True
        while not done:
            with torch.no_grad():
                q = q_net(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0))[0]
            if first:
                out["q0"].append(float(q.max()))
                first = False
            obs, r, terminated, truncated, _ = env.step(int(q.argmax()))
            rewards.append(float(r))
            done = terminated or truncated
        rewards = np.asarray(rewards)
        out["returns"].append(float(rewards.sum()))
        out["disc_returns"].append(float(np.sum(gamma ** np.arange(len(rewards)) * rewards)))
        out["terminated"].append(bool(terminated))
        out["rewards"].append(rewards)
    env.close()
    res = {k: np.asarray(v) for k, v in out.items() if k != "rewards"}
    res["rewards"] = out["rewards"]
    return res


class DQNLogger:
    """Provided. Collects what train_dqn reports, as NumPy arrays (Lab 8, Module 9).

    train_dqn calls step() once per environment step (last), update() once per gradient
    step and sync() at every target copy, then returns result(). Every log_every steps it
    records a log point; every eval_every steps (if set) a greedy evaluation; at the first
    gradient step at or after each of snapshot_steps, one batch snapshot.
    """

    def __init__(
        self,
        config,
        steps,
        q_net,
        target_net,
        make_env,
        *,
        log_every=1000,
        eval_every=None,
        eval_episodes=10,
        probe_obs=None,
        snapshot_steps=(),
        verbose=True,
    ):
        self.cfg, self.steps, self.make_env = config, steps, make_env
        self.q_net, self.target_net = q_net, target_net
        self.log_every, self.eval_every, self.eval_episodes = log_every, eval_every, eval_episodes
        if probe_obs is None:
            probe_obs = probe_states(make_env)
        self.probe = torch.as_tensor(np.asarray(probe_obs, dtype=np.float32))
        with torch.no_grad():
            self.n_actions = int(q_net(self.probe[:1]).shape[1])
        self.pending = sorted(int(s) for s in snapshot_steps)
        self.verbose = verbose
        self.print_every = max(log_every, steps // 10)
        self.start = time.perf_counter()
        self.last_sync = 0
        self.ep_ret = self.ep_disc = 0.0
        self.ep_len = 0
        self.e = {k: [] for k in ("return", "disc_return", "length", "end_step", "terminated")}
        keys = ("step", "epsilon", "loss", "td_abs_mean", "td_abs_max", "q_max_mean")
        keys += ("grad_norm", "q_probe", "q_probe_target", "q_terminal", "steps_since_sync")
        self.p = {k: [] for k in keys}
        keys = ("step", "return", "disc_return", "q0")
        self.ev = {k: [] for k in keys}
        keys = ("step", "q", "actions", "gathered", "target", "reward", "terminated")
        self.s = {k: [] for k in keys}
        self._reset_acc()

    def _reset_acc(self):
        self.acc = dict(n=0, loss=0.0, td=0.0, td_n=0, td_max=0.0, qmax=0.0, gn=0.0)
        self.acc.update(qt=0.0, qt_n=0)

    def update(self, t, batch, q_values, q_sa, y, loss, grad_norm):
        """One gradient step: q_values (B, |A|) and q_sa (B,) as the loss used them."""
        with torch.no_grad():
            td = (y - q_sa).abs()
            a = self.acc
            a["n"] += 1
            a["loss"] += float(loss)
            a["td"] += float(td.sum())
            a["td_n"] += int(td.numel())
            a["td_max"] = max(a["td_max"], float(td.max()))
            a["qmax"] += float(q_values.max(dim=1).values.mean())
            a["gn"] += float(grad_norm)
            term = batch.terminated > 0.5
            a["qt"] += float(q_sa[term].sum())
            a["qt_n"] += int(term.sum())
            while self.pending and t >= self.pending[0]:
                self.pending.pop(0)
                self.s["step"].append(t)
                self.s["q"].append(q_values.detach().numpy().copy())
                self.s["actions"].append(batch.act.numpy().copy())
                self.s["gathered"].append(q_sa.detach().numpy().copy())
                self.s["target"].append(y.detach().numpy().copy())
                self.s["reward"].append(batch.rew.numpy().copy())
                self.s["terminated"].append(batch.terminated.numpy() > 0.5)

    def sync(self, t):
        """The target network was just copied from the online network."""
        self.last_sync = t

    def step(self, t, epsilon, reward, terminated, truncated):
        """The end of environment step t (1, 2, ...), after any update and sync."""
        self.ep_disc += self.cfg.gamma**self.ep_len * float(reward)
        self.ep_ret += float(reward)
        self.ep_len += 1
        if terminated or truncated:
            for k, v in zip(
                self.e,
                (self.ep_ret, self.ep_disc, self.ep_len, t, bool(terminated)),
                strict=True,
            ):
                self.e[k].append(v)
            self.ep_ret = self.ep_disc = 0.0
            self.ep_len = 0
        if t % self.log_every == 0:
            self._log_point(t, epsilon)
        if self.eval_every and t % self.eval_every == 0:
            ev = evaluate_greedy(self.q_net, self.make_env, self.eval_episodes, self.cfg.gamma)
            for k, v in zip(self.ev, (t, ev["returns"], ev["disc_returns"], ev["q0"]), strict=True):
                self.ev[k].append(float(np.mean(v)) if k != "step" else v)
        if self.verbose and t % self.print_every == 0:
            recent = self.e["return"][-10:]
            mean = f"{np.mean(recent):6.1f}" if recent else "     -"
            q = self.p["q_probe"][-1].max(axis=1).mean() if self.p["q_probe"] else np.nan
            print(
                f"step {t:>7,} · eps {epsilon:4.2f} · last 10 returns {mean} · "
                f"max Q at probe states {q:7.2f} · {time.perf_counter() - self.start:5.1f} s"
            )

    def _log_point(self, t, epsilon):
        a, nan = self.acc, float("nan")
        n = a["n"]
        with torch.no_grad():
            q_on = self.q_net(self.probe).numpy().copy()
            q_tg = self.target_net(self.probe).numpy().copy()
        values = (
            t,
            epsilon,
            a["loss"] / n if n else nan,
            a["td"] / a["td_n"] if n else nan,
            a["td_max"] if n else nan,
            a["qmax"] / n if n else nan,
            a["gn"] / n if n else nan,
            q_on,
            q_tg,
            a["qt"] / a["qt_n"] if a["qt_n"] else nan,
            t - self.last_sync,
        )
        for k, v in zip(self.p, values, strict=True):
            self.p[k].append(v)
        self._reset_acc()

    def result(self):
        """The log: a dict of NumPy arrays, plus "config" (a dict) and "seconds"."""
        f4, i8 = np.float32, np.int64
        B, A, P = self.cfg.batch_size, self.n_actions, self.probe.shape[0]
        out = {
            "ep_return": np.asarray(self.e["return"], f4),
            "ep_disc_return": np.asarray(self.e["disc_return"], f4),
            "ep_length": np.asarray(self.e["length"], np.int32),
            "ep_end_step": np.asarray(self.e["end_step"], i8),
            "ep_terminated": np.asarray(self.e["terminated"], bool),
            "probe_obs": self.probe.numpy().copy(),
        }
        for k in ("step", "steps_since_sync"):
            out[k] = np.asarray(self.p[k], i8)
        for k in ("epsilon", "loss", "td_abs_mean", "td_abs_max", "q_max_mean", "grad_norm"):
            out[k] = np.asarray(self.p[k], f4)
        out["q_terminal"] = np.asarray(self.p["q_terminal"], f4)
        for k in ("q_probe", "q_probe_target"):
            out[k] = np.asarray(self.p[k], f4).reshape(-1, P, A)
        out["eval_step"] = np.asarray(self.ev["step"], i8)
        for k in ("return", "disc_return", "q0"):
            out[f"eval_{k}"] = np.asarray(self.ev[k], f4)
        out["snap_step"] = np.asarray(self.s["step"], i8)
        shapes = dict(q=(B, A), actions=(B,), gathered=(B,), target=(B,), reward=(B,))
        types = dict(q=f4, actions=i8, gathered=f4, target=f4, reward=f4)
        for k, shape in shapes.items():
            out[f"snap_{k}"] = np.asarray(self.s[k], types[k]).reshape(-1, *shape)
        out["snap_terminated"] = np.asarray(self.s["terminated"], bool).reshape(-1, B)
        out["config"] = dataclasses.asdict(self.cfg)
        out["seconds"] = round(time.perf_counter() - self.start, 2)
        return out


def dqn_parts(make_env, config, seed, steps, **log_options):
    """train_dqn's setup, seeded in a fixed order (torch, then a NumPy generator, then the first
    reset), so every lab's runs match the recorded ones. Returns a namespace with env, obs
    (the first state), n_actions, rng, q_net, target_net (a copy of q_net), opt (Adam),
    buffer (an empty ReplayBuffer) and log (a DQNLogger; log_options go to it)."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    env = make_env()
    obs_dim, n_actions = env.observation_space.shape[0], env.action_space.n
    q_net = make_q_net(obs_dim, n_actions, config)
    target_net = make_q_net(obs_dim, n_actions, config)
    target_net.load_state_dict(q_net.state_dict())  # w- = w
    opt = torch.optim.Adam(q_net.parameters(), lr=config.lr)
    buffer = ReplayBuffer(config.buffer_size, obs_dim)  # yours, from Lab 8 Exercise 1
    log = DQNLogger(config, steps, q_net, target_net, make_env, **log_options)
    obs, _ = env.reset(seed=seed)
    return types.SimpleNamespace(
        env=env,
        obs=obs,
        n_actions=n_actions,
        rng=rng,
        q_net=q_net,
        target_net=target_net,
        opt=opt,
        buffer=buffer,
        log=log,
    )

In [ ]:
# Reference code included from labs/shared/dqn.py.
# origin: labs/src/08-dqn.py
# The reference version of your Lab 8 code: the replay buffer (Exercise 1), the masked target
# (Exercise 2), the training loop (Exercise 3), the Double DQN target (Exercise 4) and Monte
# Carlo returns (Exercise 5). Include labs/shared/dqn_setup.py first: train_dqn calls
# dqn_parts, epsilon_at, act, gather_q, huber and dqn_target from it, each by name when it
# runs, and dqn_parts builds this file's ReplayBuffer.
class ReplayBuffer:
    """A ring buffer of transitions in preallocated NumPy arrays."""

    def __init__(self, capacity, obs_dim):
        self.capacity = capacity
        self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.act = np.zeros(capacity, dtype=np.int64)
        self.rew = np.zeros(capacity, dtype=np.float32)
        self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.truncated = np.zeros(capacity, dtype=np.float32)
        self.ptr = 0  # the slot the next transition goes to
        self.size = 0  # how many slots hold a transition

    def add(self, obs, act, rew, next_obs, terminated, truncated):
        """Write one transition at slot ptr; advance ptr and size."""
        i = self.ptr
        self.obs[i], self.act[i], self.rew[i] = obs, act, rew
        self.next_obs[i] = next_obs
        self.terminated[i], self.truncated[i] = terminated, truncated
        self.ptr = (i + 1) % self.capacity  # the oldest goes next
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, rng):
        """Uniform over the filled slots (replay sampling)."""
        idx = rng.integers(0, self.size, size=batch_size)
        return to_batch(self, idx)


def td_target(batch, q_next, gamma):
    """r + gamma (1 - terminated) q_next: truncation still bootstraps."""
    return batch.rew + gamma * (1.0 - batch.terminated) * q_next


def train_dqn(make_env, config, seed, steps, *, target_fn=None, **log_options):
    """DQN (Mnih et al. 2015, Algorithm 1). Returns (q_net, log)."""
    target_fn = target_fn or dqn_target
    p = dqn_parts(make_env, config, seed, steps, **log_options)
    obs = p.obs
    for t in range(1, steps + 1):
        eps = epsilon_at(t, steps, config)
        a = act(p.q_net, obs, eps, p.n_actions, p.rng)
        next_obs, r, terminated, truncated, _ = p.env.step(a)
        p.buffer.add(obs, a, r, next_obs, terminated, truncated)
        obs = next_obs
        if terminated or truncated:
            obs, _ = p.env.reset()
        if t > config.learning_starts and t % config.train_freq == 0:
            batch = p.buffer.sample(config.batch_size, p.rng)
            q_values = p.q_net(batch.obs)
            q_sa = gather_q(q_values, batch.act)
            y = target_fn(batch, p.q_net, p.target_net, config.gamma)
            loss = huber(y - q_sa, config.huber_threshold).mean()  # (DQN loss)
            p.opt.zero_grad()
            loss.backward()
            norm = nn.utils.clip_grad_norm_(p.q_net.parameters(), config.max_grad_norm)
            p.opt.step()
            p.log.update(t, batch, q_values, q_sa, y, loss, norm)
        if config.target_sync and t % config.target_sync == 0:
            p.target_net.load_state_dict(p.q_net.state_dict())  # w- <- w
            p.log.sync(t)
        p.log.step(t, eps, r, terminated, truncated)
    p.env.close()
    return p.q_net, p.log.result()


def double_dqn_target(batch, q_net, target_net, gamma):
    """Select with the online network, evaluate with the target."""
    with torch.no_grad():
        a_next = q_net(batch.next_obs).argmax(dim=1)  # w selects
        q_next = gather_q(target_net(batch.next_obs), a_next)  # w- values
        return td_target(batch, q_next, gamma)  # (Double DQN target)


def mc_return(rewards, gamma):
    """Returns-to-go, computed backward (MC return)."""
    g = np.zeros(len(rewards))
    total = 0.0
    for t in reversed(range(len(rewards))):
        total = rewards[t] + gamma * total
        g[t] = total
    return g

### Provided: the recorded runs, one run's log, and the plots

`prl.data.load("m09_broken_runs")` holds the healthy run and runs A to D, each on seeds 0, 1
and 2, at the live budget. `run_log(run, seed)` returns one run's log as a dict, with the
keys `train_dqn` writes. `dashboard(runs)` draws the return, the loss, the mean |δ| and Q at
the start states for the runs you name, every seed as its own line.

In [ ]:
RUNS = prl.data.load("m09_broken_runs")
GAMMA = float(RUNS["gamma"])
G_MAX = sum(GAMMA**t for t in range(500))  # the most a 500-step CartPole episode can earn


def run_log(run, seed):
    """Provided: the log of one recorded run ("healthy", "A" ... "D") and seed (0, 1, 2)."""
    s = int(np.flatnonzero(RUNS["seeds"] == seed)[0])
    log = {"step": RUNS["step"], "probe_obs": RUNS["probe_obs"]}
    for key in RUNS:
        if key.startswith(f"{run}__"):
            name = key.split("__", 1)[1]
            arr = RUNS[key]
            if name.startswith("ep_") and name != "ep_seed":
                log[name] = arr[RUNS[f"{run}__ep_seed"] == seed]
            elif name != "ep_seed":
                log[name] = arr[s]
    return log


def smooth_returns(log, every=2500):
    """Provided: the mean return of the episodes that ended in each `every`-step window."""
    edges = np.arange(every, log["step"][-1] + 1, every)
    end, ret = log["ep_end_step"], log["ep_return"]
    means = [ret[(end > e - every) & (end <= e)].mean() for e in edges]
    return edges, np.array(means)


def q_start(log):
    """Provided: Q at the start states at every log point, the mean over them of max_a Q."""
    return log["q_probe"].max(axis=-1).mean(axis=-1).astype(float)


# The healthy run is the reference, drawn as reference lines are; A-D differ by line style too.
RUN_STYLE = {
    "healthy": dict(color=prl.plot.NEUTRAL_LIGHT, ls="-"),
    "A": dict(color=prl.plot.SERIES[0], ls=(0, (4.5, 1.8))),
    "B": dict(color=prl.plot.SERIES[1], ls=(0, (1, 1.6))),
    "C": dict(color=prl.plot.SERIES[2], ls=(0, (6, 1.8, 1.4, 1.8))),
    "D": dict(color=prl.plot.SERIES[3], ls=(0, (3, 1.2, 1, 1.2))),
}


def dashboard(runs, title, panels=("return", "loss", "td", "q")):
    """Provided: the chosen panels for the runs you name, every seed as its own line."""
    draw = {
        "return": ("return (episodes per 2,500 steps)", "linear", smooth_returns),
        "loss": ("mean loss", "log", lambda g: (g["step"], g["loss"])),
        "td": ("mean |δ|", "log", lambda g: (g["step"], g["td_abs_mean"])),
        "q": ("Q at the start states", "symlog", lambda g: (g["step"], q_start(g))),
    }
    fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.6))
    for ax, panel in zip(np.atleast_1d(axes), panels, strict=True):
        ylabel, scale, xy = draw[panel]
        for run in runs:
            for k, seed in enumerate(RUNS["seeds"]):
                name = "healthy" if run == "healthy" else f"run {run}"
                kw = dict(lw=1.2, alpha=0.85, label=name if k == 0 else None)
                ax.plot(*xy(run_log(run, int(seed))), **kw, **RUN_STYLE[run])
        if panel == "q":
            ax.axhline(G_MAX, color=prl.plot.NEUTRAL_LIGHT, ls=":", lw=1, label="99.34")
        ax.set(ylabel=ylabel, yscale=scale, xlabel="env step")
    np.atleast_1d(axes)[0].legend()
    fig.suptitle(title)
    plt.show()


def second_half_median(x):
    """Provided: the median of a signal over the second half of the run."""
    return float(np.nanmedian(np.asarray(x)[len(x) // 2 :]))


def last_five(x):
    """Provided: the mean of a signal over its last five log points."""
    return float(np.nanmean(np.asarray(x)[-5:]))

## Exercise 1 · Two exact checks (9 min)

**Two exact checks.** Some properties of a correct DQN hold exactly, so a log can check them.
In each logged batch, the value the loss used for row i is Q(s_i, a_i; w), read from the same
forward pass (TD error). And the target network's values change at every sync. At three
steps the log kept one batch: `snap_q`, Q(s, ·; w) for every row (shape (S, B, |𝒜|), here
S = 3 snapshots of B = 32 rows and |𝒜| = 2 actions), `snap_actions` (S, B) and
`snap_gathered` (S, B), the value the loss used. Every 500 steps, when the target is also
synced, the log kept the target network's Q at 8 fixed start states, `q_probe_target`
(shape (L, 8, 2)). Write `gather_check` and `target_lag`. The run cell tries them on the
healthy run only, every seed.

**Predict.** On the healthy run, will `gather_check` be about 0.01, exactly 0.0, or a value that changes
with the seed? What will `target_lag` be at the last log point of the healthy run, and of a
run whose target never syncs (log points from step 500 to 50,000)?

In [ ]:
def gather_check(log):
    """The share of snapshot rows whose log["snap_gathered"] value is not Q(s_i, a_i), the
    entry of log["snap_q"] for the row's action log["snap_actions"]. Compare with
    np.isclose(..., rtol=1e-5, atol=1e-6). Returns one float in [0, 1]."""
    # TODO 1 (part 1): pick Q(s_i, a_i) for every row, then compare.
    raise NotImplementedError("TODO 1: write gather_check, or run lab.use_reference(1) to continue")


def target_lag(log):
    """Env steps since the target network's probe values last changed, at every log point:
    lag[0] = 0; for i >= 1, lag[i] = step[i] - step[j], where j <= i is the latest log point
    at which log["q_probe_target"][j] differs from [j - 1] in any entry (j = 0 if none).
    Returns an int array of shape (L,)."""
    # TODO 1 (part 2): one loop over the log points, remembering the step of the last change.
    raise NotImplementedError("TODO 1: write target_lag, or run lab.use_reference(1) to continue")

<details><summary>Hint 1</summary>

`np.take_along_axis` picks one entry per row along an axis; its index array needs as many
dimensions as `snap_q`. The mean of a boolean array is the share of True. In `target_lag`,
keep `last`, the step of the latest change, starting at `step[0]`; `np.any(a != b)` is True
if any entry differs.

</details>

In [ ]:
#@title Solution 1 · gather_check, target_lag (open after trying) { display-mode: "form" }
@lab.solution(1)
def gather_check(log):
    """Share of batch rows whose gathered value is not Q(s_i, a_i)."""
    q, a = log["snap_q"], log["snap_actions"]
    want = np.take_along_axis(q, a[..., None], axis=-1)[..., 0]
    wrong = ~np.isclose(log["snap_gathered"], want, rtol=1e-5, atol=1e-6)
    return float(wrong.mean())


@lab.solution(1)
def target_lag(log):
    """Env steps since the target network's values last changed."""
    q_tgt, step = log["q_probe_target"], log["step"]
    lag = np.zeros(len(step), dtype=np.int64)
    last = step[0]
    for i in range(1, len(step)):
        if np.any(q_tgt[i] != q_tgt[i - 1]):
            last = step[i]
        lag[i] = step[i] - last
    return lag

In [ ]:
dashboard(["healthy"], "The healthy run, 3 seeds")
for seed in RUNS["seeds"]:
    log = run_log("healthy", int(seed))
    lag = target_lag(log)
    print(
        f"healthy, seed {seed}: gather_check {gather_check(log)}; target_lag at the last log "
        f"point {lag[-1]}, largest {lag.max()} (at step {log['step'][np.argmax(lag)]:,})"
    )

**Explain.** Why is the healthy gather check exactly 0.0 and not merely small, and what would a value near
1 say about the code (TD error)? The largest lag comes early in the run: before the first
gradient step, what does a sync copy? What would a `target_lag` that grows with the step say?

In [ ]:
lab.check(1, m09.check_gather_check, gather_check, label="1.1")

In [ ]:
lab.check(1, m09.check_target_lag, target_lag, label="1.2")

## Exercise 2 · Two scale signals (10 min)

**Two scale signals.** Compare what the network believes with what the agent earns: at every
log point, Q at the start states (the mean over the 8 probe states of max_a Q(s, a; w)) minus
the mean discounted return of the episodes that ended in the last `window` env steps
(Q drift). And the TD error against the scale of Q: the mean |δ| divided by the mean over the
batch of max_a Q(s, a; w) (relative TD error). Write `q_drift` and `td_error_scale`. The run
cell tries them on the healthy run, every seed.

**Predict.** CartPole pays 1 per step and stops at 500 steps. At γ = 0.99, what is the largest discounted
return any episode can earn from its first state? On the healthy run, will the relative TD
error in the second half be about 0.001, 0.01 or 0.1?

In [ ]:
def q_drift(log, window=5000):
    """Q drift at every log point i: q_bar[i] minus the mean of log["ep_disc_return"] over the
    episodes with t - window < log["ep_end_step"] <= t, where t = log["step"][i] and q_bar =
    log["q_probe"].max(axis=-1).mean(axis=-1). NaN where no episode ended in the window.
    Returns an array of shape (L,)."""
    # TODO 2 (part 1): q_bar first; then a loop over the log points.
    raise NotImplementedError("TODO 2: write q_drift, or run lab.use_reference(2) to continue")


def td_error_scale(log):
    """The relative TD error at every log point: log["td_abs_mean"] / |log["q_max_mean"]|,
    both of shape (L,). Returns an array of shape (L,)."""
    # TODO 2 (part 2): one line.
    raise NotImplementedError(
        "TODO 2: write td_error_scale, or run lab.use_reference(2) to continue"
    )

<details><summary>Hint 2</summary>

`log["q_probe"]` has shape (L, 8, 2): a max over one axis and a mean over the next leave one
number per log point. Start the output with `np.full(L, np.nan)`, and pick the recent
episodes with a boolean mask on `log["ep_end_step"]`. NumPy divides arrays element by element.

</details>

In [ ]:
#@title Solution 2 · q_drift, td_error_scale (open after trying) { display-mode: "form" }
@lab.solution(2)
def q_drift(log, window=5000):
    """Q at the probe states minus recent discounted returns."""
    q_bar = log["q_probe"].max(axis=-1).mean(axis=-1)  # (L,)
    end, g0 = log["ep_end_step"], log["ep_disc_return"]
    drift = np.full(len(log["step"]), np.nan)
    for i, t in enumerate(log["step"]):
        recent = (end > t - window) & (end <= t)
        if recent.any():
            drift[i] = q_bar[i] - g0[recent].mean()  # (Q drift)
    return drift


@lab.solution(2)
def td_error_scale(log):
    """Mean |delta| over the batch Q scale, per log point."""
    q_scale = np.abs(log["q_max_mean"])
    return log["td_abs_mean"] / q_scale  # (relative TD error)

In [ ]:
print(f"The most any CartPole episode can earn from its first state at γ = {GAMMA}: {G_MAX:.2f}")
fig, (ax_q, ax_r) = plt.subplots(1, 2, figsize=(10, 3.6))
for k, seed in enumerate(RUNS["seeds"]):
    log = run_log("healthy", int(seed))
    drift, scale = q_drift(log), td_error_scale(log)
    print(
        f"healthy, seed {seed}: Q drift (last 5 log points) {last_five(drift):6.2f}; "
        f"relative TD error (second half, median) {second_half_median(scale):.4f}"
    )
    label = "healthy" if k == 0 else None
    ax_q.plot(log["step"], q_start(log), label=label, **RUN_STYLE["healthy"])
    ax_q.plot(
        log["step"],
        q_start(log) - drift,
        ls="--",
        color=prl.plot.SERIES[0],
        label="discounted return earned" if k == 0 else None,
    )
    ax_r.plot(log["step"], scale, **RUN_STYLE["healthy"])
ax_q.axhline(G_MAX, color=prl.plot.NEUTRAL_LIGHT, ls=":", lw=1, label=f"{G_MAX:.2f}")
ax_q.set(xlabel="env step", ylabel="Q at the start states", title="Q against the return earned")
ax_r.set(xlabel="env step", ylabel="mean |δ| / mean max Q", yscale="log", title="Relative TD error")
ax_q.legend()
plt.show()

**Explain.** On the healthy run, is Q at the start states above or below the discounted return earned,
and how close does either come to 99.34 (Q drift)? How much does the relative TD error vary
from seed to seed? Why is it unchanged when every reward is 10 times larger?

In [ ]:
lab.check(2, m09.check_q_drift, q_drift, label="2.1")

In [ ]:
lab.check(2, m09.check_td_error_scale, td_error_scale, label="2.2")

## Exercise 3 · Broken runs A to D (7 min)

**Broken runs A to D.** The teammate's four runs. Each has exactly one of these causes:

| Key | Cause |
|---|---|
| `"mask"` | The terminal mask is missing: targets bootstrap past the end of an episode. |
| `"sync"` | The target network is never updated from the online network. |
| `"lr"` | The learning rate is far too large. |
| `"gather"` | The loss reads the wrong entries of Q(s, ·), not Q(s, a) for the action taken. |
| `"reward"` | The rewards are scaled up: every reward is many times CartPole's 1. |
| `"seed"` | Nothing is wrong with the code or the configuration: an unlucky seed. |

The run cell computes your four signals on every run and seed, prints the rewards the logged
batches held, and draws the return and Q panels of all five runs. Write your diagnoses in
`diagnoses`, with the evidence for each as a comment.

**Predict.** Before you run the cell: for a run whose terminal mask is missing, which of your four signals
should leave the healthy range, and which way? For a run whose learning rate is far too
large?

In [ ]:
def diagnoses():
    """Your diagnosis of each broken run: a dict with the keys "A", "B", "C" and "D", each
    one key of the table above ("mask", "sync", "lr", "gather", "reward" or "seed")."""
    # TODO 3: return {"A": ..., "B": ..., "C": ..., "D": ...}, the evidence as comments.
    raise NotImplementedError("TODO 3: write diagnoses, or run lab.use_reference(3) to continue")

<details><summary>Hint 3</summary>

Start with the exact checks: a check that fails names its cause. For a run that passes both,
is its Q above any return an episode can earn? 'seed': how many seeds failed? 'reward': which
rewards did the logged batches hold? What is left is a setting, not a bug.

</details>

In [ ]:
#@title Solution 3 · diagnoses (open after trying) { display-mode: "form" }
@lab.solution(3)
def diagnoses():
    """What made runs A to D (see How runs A to D were made)."""
    return {
        "A": "lr",  # exact checks pass, Q in range, rewards 1, 3 of 3 fail
        "B": "gather",  # the gather check fails
        "C": "mask",  # Q far above 99.34 with rewards of 1
        "D": "sync",  # the target never changes
    }

In [ ]:
def short(v):
    """Provided: a number in at most 7 characters."""
    if abs(v) >= 1000:
        return f"{v:,.0f}"
    return f"{v:.0f}" if abs(v) >= 10 else f"{v:.3g}"


SUMMARY = {  # column -> (your signal, how one seed's curve is summarized)
    "gather_check": (gather_check, float),
    "target_lag, last": (target_lag, lambda lag: float(lag[-1])),
    "Q drift, last 5": (q_drift, last_five),
    "rel. TD error, 2nd half": (td_error_scale, second_half_median),
}
values = {
    name: {run: [summ(fn(run_log(run, int(s)))) for s in RUNS["seeds"]] for run in RUN_STYLE}
    for name, (fn, summ) in SUMMARY.items()
}
healthy_td = np.median(values["rel. TD error, 2nd half"]["healthy"])
print(f"Each cell: the range over seeds {RUNS['seeds'].tolist()}. The relative TD error also as")
print("a multiple of the healthy median.")
WIDTH = {"gather_check": 15, "target_lag, last": 17, "Q drift, last 5": 17}
print(f"{'run':8s}" + "".join(f"{name:>{WIDTH.get(name, 26)}s}" for name in SUMMARY))
for run in RUN_STYLE:
    row = []
    for name in SUMMARY:
        lo, hi = min(values[name][run]), max(values[name][run])
        cell = short(lo) if short(lo) == short(hi) else f"{short(lo)} to {short(hi)}"
        if name == "rel. TD error, 2nd half":
            cell += f" ({hi / healthy_td:.1f}x)"
        row.append(f"{cell:>{WIDTH.get(name, 26)}s}")
    print(f"{run:8s}" + "".join(row))
print(f"Other evidence (Q at the start states can reach about {G_MAX:.2f} at most):")
for run in RUN_STYLE:
    logs = [run_log(run, int(s)) for s in RUNS["seeds"]]
    seen = np.unique(np.concatenate([g["snap_reward"].ravel() for g in logs]))
    q_end = [q_start(g)[-1] for g in logs]
    final = [smooth_returns(g)[1][-1] for g in logs]
    print(
        f"  {run:8s} rewards in the batches {seen.tolist()}; Q at start, end "
        f"{short(min(q_end))} to {short(max(q_end))}; return, end {[round(x) for x in final]}"
    )
dashboard(list(RUN_STYLE), "All five runs, 3 seeds each", panels=("return", "q"))
print(f"Your diagnoses: {diagnoses()}")

**Explain.** Run B fails the gather check and is out of range on two more signals: why does one wrong
gather move them too? Which signals single out one cause, and which flag several? For run A,
which evidence rules out each of the other five causes? (Gotcha: "An out-of-range signal
names the cause.")

In [ ]:
lab.check(3, m09.check_diagnosis, diagnoses, "A", label="3.1")

In [ ]:
lab.check(3, m09.check_diagnosis, diagnoses, "B", label="3.2")

In [ ]:
lab.check(3, m09.check_diagnosis, diagnoses, "C", label="3.3")

In [ ]:
lab.check(3, m09.check_diagnosis, diagnoses, "D", label="3.4")

### How runs A to D were made

This cell shows how the teammate made runs A to D: a configuration change, or a function
swapped into `train_dqn`, never an edit of the shared code. Run it after checkpoints 3.1-3.4.

In [ ]:
def td_target_no_mask(batch, q_next, gamma):
    """Run C's td_target: it ignores terminated."""
    return batch.rew + gamma * q_next


def gather_q_dim0(q_values, actions):
    """Run B's gather_q: dimension 0 instead of 1."""
    return q_values.gather(0, actions.long().unsqueeze(1)).squeeze(1)


AGENTS = {  # run -> (configuration changes, functions swapped in)
    "healthy": ({}, {}),
    "A": (dict(lr=CONFIG.lr * 100), {}),
    "B": ({}, dict(gather_q=gather_q_dim0)),
    "C": ({}, dict(td_target=td_target_no_mask)),
    "D": (dict(target_sync=None), {}),
}


def train_agent(run, make_env, config, seed, steps, **kwargs):
    """Provided: train_dqn as run `run` was made; the reference functions come back after."""
    changes, swaps = AGENTS[run]
    saved = {name: globals()[name] for name in swaps}
    globals().update(swaps)
    try:
        return train_dqn(make_env, dataclasses.replace(config, **changes), seed, steps, **kwargs)
    finally:
        globals().update(saved)

## Exercise 4 · Probe environments (11 min)

**Probe environments.** A diagnosis from logs is a hypothesis. A probe environment tests one
piece of an agent where the right answer is known (Jones 2021). Write probe 1,
`ConstantRewardEnv` (reward +1 every episode), and probe 2, `ObsRewardEnv` (reward equal to
a random observation of +1 or −1). The run cell trains the healthy agent and the agents of
runs A to D on both.

**Predict.** Probe 1 pays 1 and ends after one step. At γ = 0.99, what Q should a correct agent learn, and
what Q would an agent that ignores `terminated` converge to? The probe run syncs the target
every 100 steps for 2,000 steps: about what Q will run C show at the end? Which runs will
fail probe 2?

In [ ]:
class ConstantRewardEnv(gym.Env):
    """Probe 1 (Jones): one action; the observation is always 0; every episode is one step
    long, pays +1 and ends with terminated=True. Spaces: a float32 Box of shape (1,) on
    [-1, 1], and Discrete(1). reset(*, seed=None, options=None) passes seed to super().reset()
    and returns (observation, info)."""

    # TODO 4 (part 1): __init__ sets the two spaces; then reset and step.
    def __init__(self):
        raise NotImplementedError(
            "TODO 4: write ConstantRewardEnv, or run lab.use_reference(4) to continue"
        )


class ObsRewardEnv(gym.Env):
    """Probe 2 (Jones): the same spaces; the observation is +1 or -1 with equal probability,
    drawn from self.np_random at reset; the reward equals it; every episode is one step long
    and ends with terminated=True; step returns the same observation."""

    # TODO 4 (part 2): keep the observation on self between reset and step.
    def __init__(self):
        raise NotImplementedError(
            "TODO 4: write ObsRewardEnv, or run lab.use_reference(4) to continue"
        )

<details><summary>Hint 4</summary>

`super().reset(seed=seed)` seeds `self.np_random`, Gymnasium's generator for the
environment, so the same seed gives the same observations. Return float32 arrays of shape
(1,), not Python floats: the agent stacks them into a batch. `step` returns five things:
observation, reward, terminated, truncated, info.

</details>

In [ ]:
#@title Solution 4 · ConstantRewardEnv, ObsRewardEnv (open after trying) { display-mode: "form" }
@lab.solution(4)
class ConstantRewardEnv(gym.Env):
    """Probe 1: observation 0, one step, reward +1."""

    def __init__(self):
        box = gym.spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        self.observation_space = box
        self.action_space = gym.spaces.Discrete(1)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return np.zeros(1, np.float32), {}

    def step(self, action):
        return np.zeros(1, np.float32), 1.0, True, False, {}


@lab.solution(4)
class ObsRewardEnv(gym.Env):
    """Probe 2: observation +1 or -1, one step, reward = observation."""

    def __init__(self):
        box = gym.spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        self.observation_space = box
        self.action_space = gym.spaces.Discrete(1)
        self.obs = 0.0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.obs = 1.0 if self.np_random.random() < 0.5 else -1.0
        return np.array([self.obs], np.float32), {}

    def step(self, action):
        return np.array([self.obs], np.float32), self.obs, True, False, {}

In [ ]:
PROBE_CONFIG = dataclasses.replace(
    CONFIG, learning_starts=64, target_sync=100, train_freq=1, eps_fraction=0.0
)
probes = {
    "probe 1": (ConstantRewardEnv, np.array([[0.0]], np.float32)),
    "probe 2": (ObsRewardEnv, np.array([[-1.0], [1.0]], np.float32)),
}
probe_q = {}
start = time.perf_counter()
print(f"Each agent on each probe: {budget.probe_steps:,} steps, target sync every 100.")
for run in AGENTS:
    cells = []
    for name, (env_cls, obs) in probes.items():
        q_net, _ = train_agent(
            run,
            env_cls,
            PROBE_CONFIG,
            RUN_SEED,
            budget.probe_steps,
            log_every=100,
            probe_obs=obs,
            verbose=False,
        )
        with torch.no_grad():
            got = q_net(torch.as_tensor(obs))[:, 0].numpy()
        probe_q[(run, name)] = got
        shown = ", ".join(f"Q({o[0]:+.0f}) = {g:8.3f}" for o, g in zip(obs, got, strict=True))
        cells.append(f"{name}: {shown}")
    print(f"  {run:8s} " + " | ".join(cells), flush=True)
print("Exact answers: probe 1, Q(+0) = 1; probe 2, Q(-1) = -1 and Q(+1) = +1.")
print(f"{len(AGENTS) * len(probes)} probe runs in {time.perf_counter() - start:.0f} s.")
lab.metric("probe_q_c", round(float(probe_q[("C", "probe 1")][0]), 2))  # room code

**Explain.** Run C's Q on probe 1 is far from both 1 and 100: count the target syncs in the run and use
(masked target) to explain its value. Which runs passed both probes, and what does a pass
rule out, and what not? (Gotcha: "The probes pass, so the agent is correct.")

**Diagnostics for this lab.** Log more than the return and the loss: the mean and largest
|δ|, Q at fixed start states from both networks, the discounted return of every episode,
and now and then one whole batch. Check the exact invariants first: the gathered value is
Q(s_i, a_i) on every row, and the target's values change at every sync. Then compare Q at the
start states with the return earned, and the relative TD error with the healthy seeds'
range. Then test what is left on probes: a correct agent gets them right to a few decimals.

In [ ]:
lab.check(4, m09.check_constant_reward_env, ConstantRewardEnv, label="4.1")

In [ ]:
lab.check(4, m09.check_obs_reward_env, ObsRewardEnv, label="4.2")

## Exercise 5 · Reporting a recorded sweep (9 min)

**A recorded sweep.** The teammate's next question: is the reference learning rate a good
one? `prl.data.load("m09_sweep")` holds 5 learning rates, 0.1, 0.3, 1, 3 and 10 times the
reference, each run on seeds 0 to 9 at the live budget, and each run's greedy return at the
end (`final_return`, shape (5, 10)). Write `iqm(x)`, the interquartile mean (IQM), and
`bootstrap_ci(x, statistic, ...)`, the percentile bootstrap interval (bootstrap CI), exactly
as described: the checkpoints compare them with `prl.evaluate.iqm` and
`prl.evaluate.bootstrap_ci` on the same inputs.

**Predict.** Will the reference (×1) have the highest IQM of the five? Will the tie rule separate it from
×0.3 and ×3? If you had run only seed 0 of each learning rate, would you rank the five the
same way as by their IQMs over 10 seeds?

In [ ]:
def iqm(x):
    """The interquartile mean (IQM) of the values in x: sort them, drop k = floor(n / 4) from
    each end (n = len(x)), and return the mean of the rest as a float. n = 10 drops 2 from each
    end and averages 6."""
    # TODO 5 (part 1): three lines.
    raise NotImplementedError("TODO 5: write iqm, or run lab.use_reference(5) to continue")


def bootstrap_ci(x, statistic=np.mean, *, level=0.95, n_boot=5000, seed=0):
    """Percentile bootstrap interval for statistic(x) (bootstrap CI): n_boot resamples of x,
    each n values drawn with replacement using np.random.default_rng(seed); statistic of each;
    returns the (1 - level) / 2 and 1 - (1 - level) / 2 quantiles (np.quantile) as two
    floats."""
    # TODO 5 (part 2): about five lines.
    raise NotImplementedError("TODO 5: write bootstrap_ci, or run lab.use_reference(5) to continue")

<details><summary>Hint 5</summary>

`np.sort(x)[k : n - k]` keeps the middle values. In `bootstrap_ci`, the indices must come
from `np.random.default_rng(seed)`, the generator `prl.evaluate` uses, so that the same seed
gives the same resamples and the checkpoint can compare the exact interval. `x[row]` with an
array of indices is one resample: n values drawn with replacement.

</details>

In [ ]:
#@title Solution 5 · iqm, bootstrap_ci (open after trying) { display-mode: "form" }
@lab.solution(5)
def iqm(x):
    """Mean of the middle values after dropping floor(n/4) per end."""
    x = np.sort(np.asarray(x, dtype=float).ravel())
    k = len(x) // 4
    return float(x[k : len(x) - k].mean())  # (IQM)


@lab.solution(5)
def bootstrap_ci(x, statistic=np.mean, *, level=0.95, n_boot=5000, seed=0):
    """Percentile bootstrap interval for statistic(x)."""
    x = np.asarray(x, dtype=float).ravel()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    boots = np.array([statistic(x[row]) for row in idx])
    tails = [(1 - level) / 2, 1 - (1 - level) / 2]
    lo, hi = np.quantile(boots, tails)  # (bootstrap CI)
    return float(lo), float(hi)

In [ ]:
SWEEP = prl.data.load("m09_sweep")
factors, final = SWEEP["lr_factor"], SWEEP["final_return"]
ref = int(np.flatnonzero(factors == 1.0)[0])
summary = []
print(f"Greedy return at {int(SWEEP['budget_steps']):,} steps, {final.shape[1]} seeds per rate:")
for f, x in zip(factors, final, strict=True):
    lo, hi = bootstrap_ci(x, iqm)
    summary.append((iqm(x), lo, hi))
    print(f"  lr x{f:<4g} IQM {iqm(x):6.1f} [{lo:6.1f}, {hi:6.1f}]  mean {x.mean():6.1f}")
    print(f"           seeds: {' '.join(f'{v:4.0f}' for v in x)}")
for i, f in enumerate(factors):
    if i != ref:
        v = compare(final[i], final[ref], names=(f"lr x{f:g}", "lr x1"))
        print(textwrap.fill(v.sentence.split(" (IQM")[0], 98, initial_indent="  "))
sweep_budget = Budget(
    steps=int(SWEEP["budget_steps"]),
    seeds=final.shape[1],
    wall_clock_s=float(SWEEP["seconds"].sum()),
    hardware=str(SWEEP["hardware"]),
)
print(textwrap.fill(sweep_budget.sentence(), 98))
by_seed0 = [f"x{factors[i]:g}" for i in np.argsort(-final[:, 0], kind="stable")]
by_iqm = [f"x{factors[i]:g}" for i in np.argsort([-s[0] for s in summary], kind="stable")]
print(f"Ranked by IQM:        {' > '.join(by_iqm)}")
print(f"Ranked by seed 0 only: {' > '.join(by_seed0)}")
fig, ax = plt.subplots()
xs = np.arange(len(factors))
spread = np.linspace(-0.2, 0.2, final.shape[1])  # one x per seed, so equal values stay apart
for i in xs:
    ax.plot(i + spread, final[i], ".", color=prl.plot.NEUTRAL_LIGHT)
centers = np.array([s[0] for s in summary])
err = np.array([[s[0] - s[1], s[2] - s[0]] for s in summary]).T
ax.errorbar(
    xs,
    centers,
    yerr=err,
    marker="o",
    capsize=3,
    color=prl.plot.SERIES[0],
    label="IQM [95% bootstrap CI]",
)
ax.plot(xs, final.mean(axis=1), "x", color=prl.plot.SERIES[1], label="mean")
ax.set(
    xticks=xs,
    xticklabels=[f"x{f:g}" for f in factors],
    xlabel="learning rate / reference",
    ylabel="greedy return at the budget",
    title="Learning-rate sweep, every seed",
)
ax.legend()
plt.show()

**Explain.** Which rates does the tie rule separate from the reference, and which tie? On which rates do
the mean and the IQM differ most, and why (IQM)? The highest IQM of five configurations is
the largest of five noisy estimates: is it likely to be too high or too low as an estimate of
that configuration's true performance? (Gotcha: "The best configuration's score is its
expected score.")

In [ ]:
lab.check(5, m09.check_iqm, iqm, label="5.1")

In [ ]:
lab.check(5, m09.check_bootstrap_ci, bootstrap_ci, label="5.2")

## Exercise 6 · One live seed (6 min)

**One live seed.** Train the reference configuration yourself, on your seat's seed, at the
lab's budget, and place your run against the recorded band of the reference configuration at
the same budget (`prl.data.band`). Your seed is one more draw from the same distribution as
the recorded seeds. Write `band_position`, which says where a value lands against a band. The
cell reports it, the IQM and interval of the recorded seeds with and without yours, your four
signals on your own log, and the budget sentence of your run.

**Predict.** About 20 seats each run one seed against this 10-seed band. About how many seats will land
outside it? Look at the band's top value first: can any seat land above a band that reaches
500? Will adding your seed move the IQM by more or by less than half the width of its
interval?

In [ ]:
def band_position(value, recorded):
    """+1 if value is above every recorded value, -1 if below every one, 0 otherwise. A value
    equal to the band's top or bottom is inside (band)."""
    # TODO 6: two comparisons with the band's ends.
    raise NotImplementedError(
        "TODO 6: write band_position, or run lab.use_reference(6) to continue"
    )

<details><summary>Hint 6</summary>

The band runs from `np.min(recorded)` to `np.max(recorded)`. Only a strict inequality puts a
value outside.

</details>

In [ ]:
#@title Solution 6 · band_position (open after trying) { display-mode: "form" }
@lab.solution(6)
def band_position(value, recorded):
    """Above (+1), below (-1) or inside (0) the band's range."""
    if value > np.max(recorded):
        return 1
    if value < np.min(recorded):
        return -1
    return 0  # (band)

In [ ]:
band = prl.data.band("m09_reference", budget)  # the recording at your budget
start = time.perf_counter()
print(f"Training the reference DQN on seed {RUN_SEED} for {budget.steps:,} steps:", flush=True)
q_live, log_live = train_dqn(
    lambda: gym.make("CartPole-v1"),
    CONFIG,
    RUN_SEED,
    budget.steps,
    log_every=budget.log_every,
    eval_every=budget.eval_every,
    eval_episodes=budget.eval_episodes,
    snapshot_steps=(CONFIG.learning_starts + 1, budget.steps // 2, budget.steps),
)
seconds_live = time.perf_counter() - start
live_return = float(log_live["eval_return"][-1])  # the band's metric at the last grid point
recorded = np.asarray(band["runs"], dtype=float)[:, -1]
lab.metric("live_return", round(live_return, 1))  # room code
print(f"Your greedy return: {live_return:.1f}; band [{recorded.min():.1f}, {recorded.max():.1f}]")
print(
    textwrap.fill(
        f"Band: {band['metric']}; {len(band['seeds'])} seeds; record {band['record']}.", 98
    )
)
live_budget = Budget(steps=budget.steps, seeds=1, wall_clock_s=seconds_live, hardware=HARDWARE)
print(textwrap.fill(live_budget.sentence() + " Single seed, not a ranking.", 98))
fig, ax = plt.subplots()
for k, curve in enumerate(band["runs"]):
    label = f"recorded seeds ({len(band['seeds'])})" if k == 0 else None
    ax.plot(band["x"], curve, "o-", ms=3, color=prl.plot.NEUTRAL_LIGHT, lw=1, label=label)
ax.plot(
    log_live["eval_step"],
    log_live["eval_return"],
    "s-",
    color=prl.plot.SERIES[0],
    lw=2.5,
    label=f"your seed {RUN_SEED}",
)
ax.set(
    xlabel="env step",
    ylabel="greedy return (10 episodes)",
    title=f"Reference DQN, {budget.steps:,} steps: your seed over the recorded band",
)
ax.legend()
plt.show()
# Your functions last, so a missing one cannot cost you the training run above.
position = band_position(live_return, recorded)
lab.metric("live_position", position)  # room code: -1 below, 0 inside, +1 above the band
print(f"Your seed is {({1: 'above', 0: 'inside', -1: 'below'})[position]} the band.")
with_you = np.append(recorded, live_return)
for label, x in (("recorded seeds", recorded), ("with yours", with_you)):
    lo, hi = bootstrap_ci(x, iqm)
    print(f"  IQM of the {len(x)} {label}: {iqm(x):.1f} [{lo:.1f}, {hi:.1f}]")
print("Your four signals on your own run:")
print(f"  gather_check {gather_check(log_live):.3f}; target_lag (last) {target_lag(log_live)[-1]}")
print(
    f"  Q drift (last 5) {last_five(q_drift(log_live)):.2f}; relative TD error (second half) "
    f"{second_half_median(td_error_scale(log_live)):.4f}"
)

**Explain.** Inside the band is not a pass, and outside is not a fail (band). How many of 20 seats would
you expect outside, and what does the band's top value change? Do your four signals look like
the healthy run's? Write the sentence you would send the teammate about the reference
configuration, with its budget. The debrief pools the room's seeds.

In [ ]:
lab.check(6, m09.check_band_position, band_position, label="6.1")

In [ ]:
lab.check(6, m09.check_live_return, live_return, label="6.2")

**Your room code.** The last cell, `lab.finish()`, prints one line that starts with
`ROOM m09`. It holds your seat, the checkpoints you passed (3.1 to 3.4 are your four
diagnoses), whether QUICK budgets ran, and the numbers the debrief pools: run C's Q on probe 1
after your probe run (`probe_q_c`, Exercise 4), your live seed's greedy return
(`live_return`, Exercise 6) and where it landed against the band (`live_position`: −1 below,
0 inside, +1 above). Paste the line where your instructor says.

In [ ]:
# GENERATED by scripts/build_notebooks.py. Prints your summary and your run record.
lab.finish()

<!-- GENERATED by scripts/build_notebooks.py from _variables.yml. Do not edit this cell. -->
**Next:** [Module 10 · Policy gradients](https://project-delphi.github.io/practical-rl/modules/10-policy-gradients.html). First, paste your room code where your instructor says.

## Stretch (optional) · beyond the core path

Nothing later depends on this section.

**Probe 3: discounting.** Jones's third probe has one action, the observation 0 and then 1,
two steps, and a reward of +1 at the end only. Write it as `TwoStepEnv` (the first step
returns observation 1, reward 0, not terminated; the second returns observation 1, reward 1,
terminated). Work out Q at both observations for a correct agent at γ = 0.99, then run the
five agents on it. Which run fails this probe that passed the first two, and why?

In [ ]:
class TwoStepEnv(gym.Env):
    """Probe 3: observation 0, then 1; two steps; reward +1 at the end only. The same spaces
    as the other probes."""

    def __init__(self):
        raise NotImplementedError(
            "Stretch: write TwoStepEnv, or run lab.use_reference('stretch') to see the solution"
        )

In [ ]:
#@title Stretch solution · TwoStepEnv (open after trying) { display-mode: "form" }
@lab.solution("stretch")
class TwoStepEnv(gym.Env):
    """Probe 3: two steps, discounting."""

    def __init__(self):
        box = gym.spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        self.observation_space = box
        self.action_space = gym.spaces.Discrete(1)
        self.t = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 0
        return np.zeros(1, np.float32), {}

    def step(self, action):
        self.t += 1
        done = self.t == 2
        return np.ones(1, np.float32), float(done), done, False, {}

In [ ]:
lab.check("stretch", m09.check_two_step_env, TwoStepEnv, label="stretch")
obs3 = np.array([[0.0], [1.0]], np.float32)
for run in AGENTS:
    q_net, _ = train_agent(
        run,
        TwoStepEnv,
        PROBE_CONFIG,
        RUN_SEED,
        budget.probe_steps,
        log_every=100,
        probe_obs=obs3,
        verbose=False,
    )
    with torch.no_grad():
        got = q_net(torch.as_tensor(obs3))[:, 0].numpy()
    print(f"  {run:8s} probe 3: Q(0) = {got[0]:7.3f}, Q(1) = {got[1]:7.3f}")
print("Exact answers: Q(0) = γ · 1 = 0.99, Q(1) = 1.")

<!-- GENERATED by scripts/build_notebooks.py from _variables.yml. Do not edit this cell. -->
**Next:** [Module 10 · Policy gradients](https://project-delphi.github.io/practical-rl/modules/10-policy-gradients.html). First, paste your room code where your instructor says.